# **Attention Vs. U-Net Image Segmentation Pipeline**
This notebook executes the comparison pipeline between standard U-Net and Attention U-Net on the Oxford-IIIT Pet dataset using the modular modules under `src/`.


In [ ]:
import os
from src.data.data_utils import extract_dataset

# Setup and extract datasets
extract_dataset("Segmentattion_dataset")


In [ ]:
import os
# Verify files are extracted
print("Images count:", len(os.listdir("images")))
print("Images sample:", os.listdir("images")[:5])
print("Annotations (trimaps) sample:", os.listdir("annotations/trimaps")[:5])


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

# Visualize a sample image and its mask
img = Image.open("images/Abyssinian_1.jpg")
mask = Image.open("annotations/trimaps/Abyssinian_1.png")

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(img)
plt.title("Sample Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(mask)
plt.title("Sample Mask")
plt.axis("off")
plt.show()


In [ ]:
from src.data.dataset import SegmentationDataset
from src.utils.visualization import visualize_random_samples

# Initialize the dataset and visualize samples
dataset = SegmentationDataset(image_dir="images", mask_dir="annotations/trimaps")
visualize_random_samples(dataset, num_samples=3)


In [ ]:
from src.data.data_utils import compute_mean_std

# Calculate dataset statistics (mean & std)
mean, std = compute_mean_std(dataset)
print("Mean:", mean)
print("Std:", std)


In [ ]:
from torch.utils.data import random_split, DataLoader
from src.data.dataset import AugmentedDataset, NormalizedDataset
import torch

# Split dataset into train and validation sets
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=generator
)

# Wrap datasets with augmentation / normalization
train_dataset = AugmentedDataset(train_dataset, mean, std)
val_dataset = NormalizedDataset(val_dataset, mean, std)

# Setup DataLoaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)


# **Part 1: Standard U-Net Baseline**
In this section, we define the standard baseline U-Net architecture, inspect its parameters, and profile its latency and computational complexity.


In [ ]:
import torch
from src.models.unet import PaddedU_Net
from src.utils.profiling import profile_model, print_layer_parameters, benchmark_latency

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Initialize baseline U-Net
model_unet = PaddedU_Net()

# Check forward pass shape sanity
x = torch.randn(1, 3, 256, 256)
print("Forward pass output shape:", model_unet(x).shape)

# Profile model parameters and MACs
profile_model(model_unet)

# Print layer-wise parameter counts
print_layer_parameters(model_unet)

# Benchmark latency
benchmark_latency(model_unet, device)


In [ ]:
from src.training.trainer import train_model
from src.models.unet import PaddedU_Net

# Train baseline U-Net model (Seed 42)
history_unet, best_val_unet = train_model(
    seed=42,
    model_class=PaddedU_Net,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=20
)


# **Part 2: Attention U-Net Model**
In this section, we define the Attention U-Net architecture, verify Attention Gate dimensions, and run parameter and latency profiles.


In [ ]:
import torch
from src.models.attention_unet import AttentionUNet
from src.utils.profiling import profile_model, print_layer_parameters, benchmark_latency

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Initialize Attention U-Net
model_att = AttentionUNet()

# Check forward pass shape sanity
x = torch.randn(1, 3, 256, 256)
print("Forward pass output shape:", model_att(x).shape)

# Profile model parameters and MACs
profile_model(model_att)

# Print layer-wise parameter counts
print_layer_parameters(model_att)

# Benchmark latency
benchmark_latency(model_att, device)


In [ ]:
from src.training.trainer import train_model
from src.models.attention_unet import AttentionUNet

# Train Attention U-Net model (Seed 42)
history_att, best_val_att = train_model(
    seed=42,
    model_class=AttentionUNet,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=20
)
